<a href="https://colab.research.google.com/github/sunonmountain/Revenue-Integrity-Engine/blob/main/Revenue_Integrity_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
from datetime import datetime

class CRMDataLifecycle:
    def __init__(self, file_path):
        self.df = pd.read_csv(file_path)
        self.audit_results = {}
        self.cleaned_df = None

    def profile_revenue_leaks(self):
        """Phase 1: Identify errors and potential revenue loss."""
        self.audit_results['total_records'] = len(self.df)
        self.audit_results['missing_emails'] = self.df['Email'].isna().sum()
        self.audit_results['missing_titles'] = self.df['Job_Title'].isna().sum()

        # Identify 'High Value Gaps': Rev > 5M and missing Title
        high_val_gaps = self.df[(self.df['Annual_Revenue_Est'] > 5000000) & (self.df['Job_Title'].isna())]
        self.audit_results['high_value_gaps'] = len(high_val_gaps)

        print("--- Profiling Complete: Revenue Leaks Identified ---")
        for key, value in self.audit_results.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
        return self.audit_results

    def clean_data(self):
        """Phase 2: Automated remediation of data errors."""
        df_c = self.df.copy()

        # 1. Standardize Names & Company
        df_c['First_Name'] = df_c['First_Name'].str.strip().str.title()
        df_c['Last_Name'] = df_c['Last_Name'].str.strip().str.title()
        df_c['Company'] = df_c['Company'].replace({'TECHFLOW': 'TechFlow Solutions'}).str.title()

        # 2. UK Phone Formatting
        def format_uk_phone(phone):
            if pd.isna(phone): return "REMEDIATION REQUIRED"
            digits = re.sub(r'\D', '', str(phone))
            if digits.startswith('44'): digits = '0' + digits[2:]
            return f"{digits[:5]} {digits[5:]}" if len(digits) == 11 else "INVALID"

        df_c['Phone'] = df_c['Phone'].apply(format_uk_phone)

        # 3. Flag Duplicates
        df_c['Is_Duplicate'] = df_c.duplicated(subset=['Email'], keep=False)

        self.cleaned_df = df_c
        print("\n--- Cleaning Complete: Data Standardized ---")
        return self.cleaned_df

    def validate_and_report(self):
        """Phase 3: Quantify ROI and data health improvements."""
        if self.cleaned_df is None:
            return "Run clean_data() first."

        # Calculate Health Score
        clean_records = self.cleaned_df[(self.cleaned_df['Phone'] != "INVALID") &
                                       (self.cleaned_df['Is_Duplicate'] == False)].shape[0]
        health_score = (clean_records / len(self.df)) * 100

        print(f"\n--- Validation & ROI Report ---")
        print(f"Initial Health Score Estimate: {100 - (self.audit_results['missing_emails']/len(self.df)*100):.2f}%")
        print(f"Final Data Health Score: {health_score:.2f}%")
        print(f"Remediation: Fixed casing, standardized UK phones, and flagged duplicates.")

        # Exporting for audit trail
        output_file = f"Cleaned_CRM_Data_{datetime.now().strftime('%Y%m%d')}.csv"
        self.cleaned_df.to_csv(output_file, index=False)
        print(f"Results exported to: {output_file}")

# 1. Initialize the framework with your file
# Make sure 'Week1_Raw_Data_Portfolio.csv' is uploaded to Colab first!
framework = CRMDataLifecycle('Week1_Raw_Data_Portfolio.csv')

# 2. Run Phase 1: The Audit
# This will print the "Revenue Leaks" to your screen
framework.profile_revenue_leaks()

# 3. Run Phase 2: The Cleaning
# This processes the data in the background
framework.clean_data()

# 4. Run Phase 3: Validation
# This prints the final health score and exports the file
framework.validate_and_report()

--- Profiling Complete: Revenue Leaks Identified ---
Total Records: 100
Missing Emails: 26
Missing Titles: 10
High Value Gaps: 4

--- Cleaning Complete: Data Standardized ---

--- Validation & ROI Report ---
Initial Health Score Estimate: 74.00%
Final Data Health Score: 8.00%
Remediation: Fixed casing, standardized UK phones, and flagged duplicates.
Results exported to: Cleaned_CRM_Data_20260313.csv
